# Exercises XP: Prompt Engineering

**What you'll learn:**
- how to analyse and improve vague prompts;
- role prompting, constraint prompting and tone specification;
- how to structure multi-part prompts for specific use cases.

**How this notebook works:** every prompt is written in a Markdown/code cell **and executed** on an open instruction-tuned model (`Qwen/Qwen2.5-1.5B-Instruct`, free, runs on Colab CPU or GPU, no API key). That way the vague and improved prompts can be compared on real outputs. The same prompts can be pasted into ChatGPT; a stronger model will follow the constraints even more closely.

In [1]:
!pip install -q "transformers<5" accelerate

In [2]:
import os, re, textwrap, time
os.environ["USE_TF"] = "0"
import torch
from transformers import pipeline, set_seed
set_seed(0)

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
llm = pipeline("text-generation", model=MODEL_ID, device_map="auto",
               torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32)

def run(prompt, system=None, max_new_tokens=300, show=True):
    msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": prompt}]
    t0 = time.time()
    out = llm(msgs, max_new_tokens=max_new_tokens, do_sample=False)[0]["generated_text"][-1]["content"].strip()
    if show:
        print(out)
        print(f"\n[{len(out.split())} words, {time.time()-t0:.0f}s]")
    return out

print("Model:", MODEL_ID, "| device:", llm.device)

Model: Qwen/Qwen2.5-1.5B-Instruct | device: cpu


## 🌟 Exercise 1: Rewrite and optimize a vague prompt
> Vague prompt: **"Write something about productivity tips."**

### 3 critical issues
1. **No brand, product or goal.** FlowNest is never mentioned, so the output cannot promote the app, and nothing says the goal is downloads or sign-ups. The model writes a generic blog-style text that any company could publish.
2. **No target audience or brand voice.** Who is it for (remote workers? students? managers?) and how should FlowNest sound (expert, warm, playful)? Without this the tone is bland and the tips are too generic to resonate with anyone.
3. **No platform, format or engagement constraints.** LinkedIn rewards a strong hook in the first line, short scannable paragraphs or bullets, a clear call-to-action, a few relevant hashtags and a moderate length. "Something" gives no length, no structure and no CTA, so the result is not ready to publish.

In [3]:
vague_1 = "Write something about productivity tips."
out_vague_1 = run(vague_1, max_new_tokens=250)

Productivity is the ability to accomplish tasks efficiently and effectively within a given timeframe. It's an essential skill for success in both personal and professional life. Here are some effective productivity tips that can help you improve your efficiency:

1. Set clear goals: Define what you want to achieve and set specific, measurable, achievable, relevant, and time-bound (SMART) goals.

2. Prioritize tasks: Identify the most important tasks first and tackle them before moving on to less critical ones.

3. Use time management techniques: Techniques such as the Pomodoro Technique or the Eisenhower Matrix can help you manage your time more effectively.

4. Eliminate distractions: Minimize interruptions and focus on completing tasks without being disturbed.

5. Take breaks: Regular breaks can help you stay focused and increase productivity over long periods of work.

6. Stay organized: Keep your workspace tidy and use tools like calendars, task lists, and reminders to keep track o

### Improved prompt (role + instructions + format/tone/length constraints + reusable line)

In [4]:
improved_1 = """Act as a senior B2B social-media copywriter for FlowNest, a focus app that blocks distractions,
schedules deep-work sessions and tracks focus time. Our brand voice is warm, expert and encouraging, never pushy.

Write a LinkedIn post for busy remote professionals and team leads who struggle with constant notifications.
- Start with a one-line hook (a question or a surprising fact) that makes people stop scrolling.
- Give exactly 3 actionable productivity tips, each as a bullet point starting with an emoji and a bold verb.
- Show naturally how FlowNest helps with at least one of the tips (no hard sell).
- End with a clear call-to-action inviting readers to try FlowNest free for 14 days, then 3 relevant hashtags.
- Length: 120-180 words. Short sentences, no jargon.

Use this format whenever we promote productivity tools on social media."""
out_improved_1 = run(improved_1, max_new_tokens=350)

**Hook:** 📅 **Are you tired of being interrupted all day?** 🧵

**Actionable Tip #1:** 💡 **Turn off your phone during work hours!**
Just set it aside when you're focused and get back to it later. It's like giving yourself a break from the digital world!

**Actionable Tip #2:** ⚙️ **Create a dedicated workspace where only work matters!**
This way, you can stay fully immersed in tasks without any distractions. Plus, it boosts your concentration levels!

**Actionable Tip #3:** 🌟 **Track your progress using FlowNest’s built-in timer!**
It shows you how much time you spend on deep work versus checking emails or browsing social media. This helps you make better decisions about what truly matters.

**Call-to-Action:** Try FlowNest today for FREE for 14 days! Sign up now [HERE](https://flownest.com/signup). Unlock your full potential with our productivity tool.

#FocusBoost #ProductivityTips #RemoteWork #DeepWork #FlowNest

[140 words, 42s]


In [5]:
def check_post(text):
    return {"mentions FlowNest": "flownest" in text.lower(),
            "bullet points": len(re.findall(r"^\s*(?:[-*•]|\d+\.|\S\s*\*\*)", text, flags=re.M)),
            "hashtags": len(re.findall(r"#[A-Za-z]\w+", text)),
            "call-to-action (try/free/download)": bool(re.search(r"\b(try|free|download|sign up|start)\b", text, re.I)),
            "words": len(text.split())}
import pandas as pd
pd.DataFrame({"vague prompt": check_post(out_vague_1), "improved prompt": check_post(out_improved_1)})

,vague prompt,improved prompt
mentions FlowNest,False,True
bullet points,9,5
hashtags,0,5
call-to-action (try/free/download),False,True
words,202,140


**Result.**
- The **vague prompt** produced a generic 202-word listicle (9 numbered tips, from SMART goals to sleep) that never mentions FlowNest, has no hook, no call-to-action and no hashtags. It could not be published as a FlowNest post.
- The **improved prompt** produced a LinkedIn-ready post (140 words, within the requested 120–180):
  - a question hook;
  - exactly 3 tips with emoji and bold verbs;
  - FlowNest woven into tip 3 ("FlowNest's built-in timer");
  - the 14-day free-trial CTA and hashtags.
- One deviation: it gave 5 hashtags instead of 3 and invented a sign-up URL. A final human review is still needed, and the rule could be tightened ("exactly 3 hashtags, no links").

**What each technique did:** the *role* set the voice (B2B copywriter, warm and expert); the *instructions* set the content (3 actionable tips, product mention, CTA); the *constraints* set the shape (hook, bullets, length). The last line ("Use this format whenever we promote productivity tools on social media") turns the prompt into a reusable **template**.

## 🌟 Exercise 2: Multi-part prompt for quiz generation
We use a short sample article on volcanic eruptions as the teacher's "uploaded" text.

In [6]:
article_volcano = """Volcanoes form where molten rock, called magma, rises from deep inside the Earth. Magma collects in a
magma chamber under the volcano. When pressure from gases such as water vapour and carbon dioxide builds up, the magma
pushes through cracks and erupts at the surface, where it is called lava. Some eruptions are explosive and throw ash,
gas and rock fragments high into the sky, like Mount St. Helens in 1980. Others are effusive: lava flows out slowly,
as in Hawaii. Most volcanoes are found along the edges of tectonic plates, especially around the Pacific Ocean in an
area called the Ring of Fire. Scientists called volcanologists monitor earthquakes, gas emissions and ground swelling
to predict eruptions and keep people safe. Volcanic ash can be dangerous, but over time it also makes soil very fertile."""
print(len(article_volcano.split()), "words")

135 words


In [7]:
vague_2 = f"Make a quiz for kids about this article.\n\n{article_volcano}"
out_vague_2 = run(vague_2, max_new_tokens=350)

Sure! Here's a fun quiz for kids based on the information provided:

---

**Quiz: Volcano Adventure**

1. **True or False:** Volcanoes form when hot rocks called magma rise from deep inside the Earth.
   - A) True
   - B) False

2. **What does magma collect in before it erupts?**
   - A) In a big hole underground
   - B) Underneath the volcano

3. **When magma pushes through cracks and erupts, what do we call it?**
   - A) Lava
   - B) Ash

4. **Which type of eruption throws ash, gas, and rock fragments high into the sky?**
   - A) Explosive
   - B) Effusive

5. **Where are most volcanoes found?**
   - A) At the bottom of the ocean
   - B) Along the edges of tectonic plates, especially around the Pacific Ocean

6. **What do scientists who study volcanoes call themselves?**
   - A) Geologists
   - B) Astronomers

7. **How do scientists know if there might be an upcoming eruption?**
   - A) By listening to music
   - B) By monitoring earthquakes, gas emissions, and ground swelling

8. **

In [8]:
improved_2 = f"""You are an experienced 7th-grade science teacher who creates fun, clear classroom materials.

TASK: Using ONLY the article below, create content for a Google Slides lesson for students aged 11-13.

Part 1 - Summary: Summarize the article in exactly 2 simple bullet points (max 20 words each).
Part 2 - Quiz: Write exactly 3 multiple-choice questions about facts stated in the article.
  - Each question has 3 options (A, B, C): 1 correct answer and 2 plausible but clearly wrong distractors.
  - Mark the correct answer on a separate line: "Answer: <letter>".

STYLE: friendly, encouraging tone; simple everyday vocabulary; short sentences; explain any science word you use.

FORMAT (Google Slides ready - one block per slide, no long paragraphs):
Slide 1 - Title: <fun title with one emoji>
Slide 2 - Summary: <2 bullets>
Slide 3 - Question 1 ... Slide 5 - Question 3 (question, options A/B/C, Answer)

ARTICLE:
\"\"\"{article_volcano}\"\"\""""
out_improved_2 = run(improved_2, max_new_tokens=450)

**Title:** Volcano Fun Facts

**Summary:**
- Volcanoes form when magma rises from deep inside the Earth.
- Magma collects in a magma chamber before erupting.

---

**Question 1:**
What happens to magma before it erupts?
a) It stays underground quietly.
b) It pushes through cracks and erupts.
c) It melts rocks all around.

**Answer: b)**

---

**Question 2:**
How do some volcanic eruptions look?
a) Quietly without much noise.
b) Explosive, throwing ash and debris high into the air.
c) Slowly flowing lava without explosions.

**Answer: b)**

---

**Question 3:**
Where are most volcanoes located?
a) In the middle of continents.
b) Along the edges of tectonic plates.
c) Near large bodies of water only.

**Answer: b)**

[117 words, 37s]


In [9]:
def check_quiz(text):
    return {"'Slide' blocks": len(re.findall(r"slide\s*\d", text, re.I)),
            "questions (lines ending with ?)": len(re.findall(r"\?\s*$", text, re.M)),
            "options A/B/C": len(re.findall(r"^\s*\(?[ABC][\).:]", text, re.M)),
            "answer keys": len(re.findall(r"answer\s*:", text, re.I)),
            "words": len(text.split())}
pd.DataFrame({"vague prompt": check_quiz(out_vague_2), "improved prompt": check_quiz(out_improved_2)})

,vague prompt,improved prompt
'Slide' blocks,0,0
questions (lines ending with ?),0,3
options A/B/C,0,0
answer keys,0,3
words,221,117


**Comparison: vague vs multi-part prompt.**
- The vague prompt (*"Make a quiz for kids about this article"*) produced **10 two-option questions** (several true/false), no summary, no answer key, and a truncated last question.
- The multi-part prompt produced:
  - a title;
  - a **2-bullet summary**;
  - **exactly 3 questions with 3 options (a/b/c)** and an **answer key** each;
  - simple vocabulary;
  - `---` separators that map naturally to slides.

**Three key ways the structured prompt prevents irrelevant or low-quality output:**
1. **Explicit scope and source grounding.** "Using ONLY the article" plus exact counts (2 bullets, 3 questions, 1 correct + 2 distractors) remove the guesswork about quantity and keep the questions tied to facts in the text. The vague prompt chose 10 questions and a different format by itself.
2. **The audience and tone are defined.** "Students aged 11–13, friendly tone, simple vocabulary, explain science words" controls the reading level. "Kids" could mean 5-year-olds or teenagers.
3. **An output format is imposed for the medium.** The slide-by-slide template with a separate "Answer:" line makes the output directly usable in Google Slides and gradable. Without it, answers are missing and the layout must be redone by hand.

**Remaining issues found by testing:** the model did not write the "Slide N" labels and **put the correct answer in position b) for all 3 questions**, an exploitable pattern. Next iteration: *"Label each block 'Slide N'"* and *"Vary the position of the correct answer (use A, B and C once each)"*.

## 🌟 Exercise 3: Add context, get better results
A sample monthly finance report stands in for the intern's input.

In [10]:
finance_report = """Monthly Finance Report - March 2025 (Acme Retail).
Revenue reached $4.2M, up 8% from February ($3.89M) and 12% year-over-year, driven by the spring online campaign;
e-commerce now represents 41% of sales (vs 35% a year ago). Gross margin fell from 46% to 43% because of higher
shipping costs (+18%) and promotional discounts. Operating expenses were $1.35M, 5% over budget, mainly due to two
new hires in customer support and a one-off $60K software migration. EBITDA was $0.46M (11% margin, target 13%).
Cash at month end: $2.1M; inventory days rose from 52 to 61 as spring stock arrived early. Accounts receivable
improved: DSO down from 38 to 33 days. Outlook: Q2 revenue forecast $13M; management plans to renegotiate the
shipping contract in April (expected savings $40K/month) and reduce discounting."""
out_vague_3 = run(f"Summarize this report.\n\n{finance_report}", max_new_tokens=250)

**Summary of Acme Retail's Monthly Finance Report for March 2025**

- **Revenue**: $4.2 million, an increase of 8% from February ($3.89 million) and 12% year-over-year.
- **Sales Mix**: E-commerce now accounts for 41% of total sales (compared to 35% a year ago), reflecting the success of the spring online campaign.
- **Gross Margin**: Declined slightly to 43%, primarily due to increased shipping costs (+18%) and promotional discounts.
- **Operating Expenses**: Totalled $1.35 million, exceeding budgeted by 5%. The main contributors include hiring new customer support staff and a one-time software migration cost of $60,000.
- **EBITDA**: Achieved $0.46 million with a 11% profit margin, aiming for 13%.
- **Financial Health**: Cash balance stood at $2.1 million, but inventory days extended to 61 days due to early arrival of spring stock.
- **Outlook**: Revenue forecasts for Q2 are set at $13 million. Management is planning to renegotiate

[146 words, 46s]


### 1. Missing context

| Context type | Missing? | What should be added? |
|---|---|---|
| **Role** | Yes | "Act as a financial analyst preparing the monthly update", so the model prioritises the numbers that matter (margin, EBITDA, cash) the way an analyst would. |
| **Audience** | Yes | A non-technical executive team: plain language, no accounting jargon, focus on business impact. |
| **Purpose** | Yes | A 3-minute spoken segment of the monthly finance update: highlight performance vs targets, risks and next actions. |
| **Input source** | Partly (the text is pasted, but its nature is not stated) | Say it is the March 2025 monthly report and that only figures from it may be used (no outside data, no invented numbers). |
| **Format/Style** | Yes | Headline, then 3 key takeaways as bullets (each with supporting figures and the change vs last month or target), then 1–2 next steps; confident, neutral tone. |
| **Constraints** | Yes | Max ~250 words (≈ 3 minutes spoken); exactly 3 takeaways; every takeaway includes at least one figure; flag misses vs budget/target explicitly. |

### 2. Rewritten prompt (6 context types)

In [11]:
improved_3 = f"""Act as a financial analyst at Acme Retail. [ROLE]
You are preparing a 3-minute spoken summary for the monthly finance update meeting. [PURPOSE]
The audience is the non-technical executive team: use plain language and explain what each number means for the business. [AUDIENCE]
Use only the figures in the March 2025 monthly report below; do not add or estimate any number that is not in it. [INPUT SOURCE]

Format: [FORMAT/STYLE]
1. One-sentence headline on overall performance.
2. Exactly 3 key takeaways as bullet points, each with supporting data (value + change vs last month, last year or target).
3. "Next steps": 2 short bullets.
Constraints: maximum 200 words; every takeaway must contain at least one figure; state clearly where we missed budget or target. [CONSTRAINTS]

REPORT:
\"\"\"{finance_report}\"\"\""""
out_improved_3 = run(improved_3, max_new_tokens=350)

**Overall Performance:** Revenue increased by 8% compared to February, reaching $4.2 million, marking a 12% growth year-over-year.

**Key Takeaways:**
1. **Sales Growth:** Sales grew by 8%, driven primarily by an increase in e-commerce sales, which now account for 41% of total sales, up from 35% a year ago.
2. **Gross Margin Decline:** Despite the rise in revenue, gross margins decreased slightly to 43%, reflecting higher shipping costs and promotional discounts.
3. **Cost Overruns:** Operating expenses exceeded expectations by 5%, largely due to new hires in customer support and a one-time software migration cost of $60,000.

**Next Steps:**
1. **Review Shipping Costs:** Negotiate the shipping contract in April to potentially save $40,000 per month.
2. **Reduce Promotional Discounts:** Implement more targeted promotions to improve profitability without sacrificing sales volume.

**Missed Budgets/Trends:**
- The company overspent operating expenses by 5%, highlighting the need for bett

In [12]:
MULT = {"k": 1e3, "thousand": 1e3, "m": 1e6, "million": 1e6}
def numbers(text):
    # normalise "$4.2M", "4.2 million", "$40K", "$40,000" -> 4200000.0 / 40000.0 so that reformatted figures still match
    vals = set()
    for num, unit in re.findall(r"(\d[\d,]*(?:\.\d+)?)\s*(million|thousand|[mk](?![a-z]))?", text, flags=re.I):
        v = float(num.replace(",", ""))
        vals.add(round(v * MULT.get(unit.lower(), 1), 2))
    return vals
report_numbers = numbers(finance_report)
for name, out in [("vague", out_vague_3), ("improved", out_improved_3)]:
    nums = numbers(out)
    print(f"{name:8s}: {len(out.split())} words | {len(nums & report_numbers)} figures from the report | "
          f"{len(nums - report_numbers)} figures NOT in the report: {sorted(nums - report_numbers)[:8]}")

vague   : 146 words | 19 figures from the report | 0 figures NOT in the report: []
improved: 169 words | 11 figures from the report | 2 figures NOT in the report: [1.0, 3.0]


**Result.**
- On this clean, short report the vague prompt already produced a decent bullet list with many figures (19 figures, all from the report), but it was a flat list of 7 items, cut off at the end, with no headline, no prioritisation and no actions. That is not a 3-minute executive talk.
- The context-rich prompt produced the requested structure: a headline, **3 key takeaways with supporting data** (revenue +8 % to $4.2M, gross margin down to 43 %, opex 5 % over budget), **next steps** (renegotiate shipping for $40K/month savings; reduce discounting), and an explicit "missed budget" section. It fits the 3-minute slot (169 words).
- The figure check finds no invented figures. The only "new" numbers (1, 3) are list numbering, and "$40,000" correctly matches the report's "$40K".
- Two gaps for the next iteration: it did not flag the **EBITDA margin miss (11 % vs 13 % target)** and slightly overran 150 words. Adding *"Always compare EBITDA and margins to target"* and a stricter word limit would fix this.

## 🌟 Exercise 4: Match prompt to purpose
**Chosen style: Structured.** It is used for a customer-support chatbot template that handles *"Where is my order?"* and return/refund requests.

In [13]:
support_system = """You are "ShopBot", the customer support assistant of TrendCart, an online fashion store.
Always answer using EXACTLY this structure:

**1. Summary of your request:** <one sentence restating the customer's issue>
**2. What I found:** <facts from the ORDER DATA only; if information is missing, say "I don't have this information">
**3. Next steps:** <numbered list of max 3 concrete steps the customer or TrendCart will take>
**4. Need more help?** <one line: offer a human agent at support@trendcart.com or chat 9am-6pm>

Rules: be polite and concise (max 120 words), never invent order details, policies or dates,
never ask for card numbers or passwords. Return policy: 30 days from delivery, items unworn, free return label."""

customer_msg = """Hi, I ordered a jacket (order #TC-48213) two weeks ago and it still hasn't arrived. Also, if it doesn't fit, can I return it?
ORDER DATA: order #TC-48213, status: shipped on 2 days ago, carrier: DHL, tracking: DH8812345, estimated delivery: tomorrow."""
out_support = run(customer_msg, system=support_system, max_new_tokens=300)

**1. Summary of your request:** You ordered a jacket with order #TC-48213 that has not yet arrived and you're unsure about returning it due to sizing issues.
**2. What I found:** The jacket was shipped on 2 days ago via DHL, with tracking number DH8812345, and is expected to arrive tomorrow.
**3. Next steps:** 
   - Check the current status of your order through the TrendCart app or website.
   - If the jacket hasn’t arrived by tomorrow, contact our customer service team at support@trendcart.com or use the chat feature available 9am-6pm daily.
   - Request a replacement or refund based on the size mismatch.
**4. Need more help?** Reach out to us anytime!

[111 words, 31s]


In [14]:
sections = ["Summary of your request", "What I found", "Next steps", "Need more help"]
print({s: (s.lower() in out_support.lower()) for s in sections}, "| words:", len(out_support.split()))

{'Summary of your request': True, 'What I found': True, 'Next steps': True, 'Need more help': True} | words: 111


**Style chosen: Structured.**

**Two key features that make it Structured:**
1. A **fixed, numbered output template** with four labelled sections (*Summary of your request → What I found → Next steps → Need more help?*) that must appear in every answer. All four were present in the test output.
2. **Explicit rules and limits** (max 120 words, a numbered list of max 3 steps, use only ORDER DATA, never ask for card numbers) that make every answer follow the same predictable shape, regardless of the customer's wording.

**Why Structured beats the other styles here:**
- Support answers must be **consistent, scannable and safe**: customers want the status and the next action fast, agents and QA teams need answers they can audit, and the format makes it easy to plug in order data or hand off to a human.
- *Conversational* would feel warmer but drift in length and content and is harder to control.
- *Exploratory* (brainstorming) is the opposite of what a customer with a late parcel needs.
- *Functional* fits single transformations (e.g. summarising a ticket), not a full customer reply.

**Observed limitation:** the model added a step not grounded in the policy ("Request a replacement or refund based on the size mismatch") instead of stating the 30-day free-return policy it was given, and used bullets rather than a numbered list. The fix: *"Quote the return policy when the customer asks about returns"*, plus an output check before sending.

## 🌟 Exercise 5: Prompt refinement challenge (style, structure and length)
**PulseOne Mini** constraints: friendly tone, bullet points, max 50 words in total, must mention battery life, fitness tracking and Bluetooth compatibility.

In [15]:
FEATURES = {"battery life": r"battery", "fitness tracking": r"fitness|track", "Bluetooth compatibility": r"bluetooth"}
FRIENDLY = r"!|you|your|love|meet|say hello|easy|😊|🎉|💪|⌚|🔋"

def evaluate_blurb(text):
    words = len(re.findall(r"[A-Za-z0-9'’-]+", text))
    bullets = [l for l in text.splitlines() if re.match(r"^\s*([-*•]|\d+\.)", l) or re.match(r"^\s*[^\w\s]", l)]
    res = {"words": words, "≤ 50 words": words <= 50, "bullet lines": len(bullets)}
    for name, pat in FEATURES.items():
        res[f"mentions {name}"] = bool(re.search(pat, text, re.I))
    res["friendly markers"] = len(re.findall(FRIENDLY, text, re.I))
    res["PASS"] = res["≤ 50 words"] and all(res[f"mentions {n}"] for n in FEATURES) and res["bullet lines"] >= 3
    return res

In [16]:
prompt_5_v1 = """Write a friendly product blurb for our new smartwatch, PulseOne Mini, for an email campaign.
Use bullet points, keep it under 50 words, and mention battery life, fitness tracking and Bluetooth compatibility."""
out_5_v1 = run(prompt_5_v1, max_new_tokens=150)
evaluate_blurb(out_5_v1)

Introducing PulseOne Mini - your personal health companion on the go! Stay connected with seamless Bluetooth pairing. Track your daily steps, calories burned, and sleep quality effortlessly. With its compact design, you can wear it anywhere without any hassle. Enjoy up to 12 hours of battery life, so you're always ready when you need to be. Upgrade your wellness journey today with PulseOne Mini!

[64 words, 15s]


{'words': 64,
 '≤ 50 words': False,
 'bullet lines': 0,
 'mentions battery life': True,
 'mentions fitness tracking': True,
 'mentions Bluetooth compatibility': True,
 'friendly markers': 8,
 'PASS': False}

In [17]:
prompt_5_v2 = """You are a friendly email copywriter for a wearable-tech brand.
Write a product blurb for the PulseOne Mini smartwatch.

STRICT RULES - follow all of them:
1. Output ONLY the blurb: no title, no introduction, no explanation, no sign-off.
2. One short opening line (max 8 words), then EXACTLY 3 bullet points starting with "- ".
3. Bullet 1 is about battery life, bullet 2 about fitness tracking, bullet 3 about Bluetooth compatibility.
4. Do not exceed 50 words in total (count every word). Each bullet: max 10 words.
5. Tone: friendly, upbeat, speak directly to the reader ("you"). One emoji per bullet is allowed.
6. Do not invent numbers or specifications."""
out_5_v2 = run(prompt_5_v2, max_new_tokens=120)
evaluate_blurb(out_5_v2)

- Stay active all day long with up to 7 days of battery life  
- Track your steps, heart rate, and more with ease  
- Connect seamlessly via Bluetooth for hands-free calls

[32 words, 8s]


{'words': 32,
 '≤ 50 words': True,
 'bullet lines': 3,
 'mentions battery life': True,
 'mentions fitness tracking': True,
 'mentions Bluetooth compatibility': True,
 'friendly markers': 1,
 'PASS': True}

**Revision v3.** v2 respects the hard constraints, but the model **invented specifications** that nobody gave it ("up to 7 days", "hands-free calls"), despite rule 6. A marketing email must not promise features the product may not have. Rule 6 alone is not enough; the fix is to **provide the facts** and forbid anything else, and to ask explicitly for a warmer, more engaging tone.

In [18]:
prompt_5_v3 = """You are a friendly, upbeat email copywriter for a wearable-tech brand.
Write a product blurb for the PulseOne Mini smartwatch using ONLY these facts:
- Battery: up to 5 days on a single charge
- Fitness: tracks steps, heart rate, sleep and 20 workout modes
- Connectivity: Bluetooth 5.3, works with iPhone and Android

STRICT RULES:
1. Output ONLY the blurb (no title, no explanation).
2. One opening line of max 8 words that speaks to the reader ("you"), then EXACTLY 3 bullet points starting with "- ".
3. Bullet 1 = battery life, bullet 2 = fitness tracking, bullet 3 = Bluetooth compatibility. Use only the facts above.
4. Do not exceed 50 words in total. Each bullet: max 12 words, starting with one emoji.
5. Friendly, energetic tone: use "you/your" and end the opening line with "!"."""
out_5_v3 = run(prompt_5_v3, max_new_tokens=120)
evaluate_blurb(out_5_v3)

You'll never need to recharge again! 🌟 Up to 5 days on a single charge! 💪 Track your steps, heart rate, sleep & 20 workouts. 🏋️‍♂️ Connect with iPhone or Android friends anytime! 🔒

[34 words, 11s]


{'words': 29,
 '≤ 50 words': True,
 'bullet lines': 0,
 'mentions battery life': False,
 'mentions fitness tracking': True,
 'mentions Bluetooth compatibility': False,
 'friendly markers': 6,
 'PASS': False}

**Revision v4: show, don't (only) tell.** With a 1.5 B model, piling up rules (v3) can make things *worse*. A **one-shot example** of the exact output format, plus the facts and fixed bullet openers, is often the most reliable constraint.

In [19]:
prompt_5_v4 = """Write a friendly product blurb in the SAME format as the example: one short opening line, then exactly 3 bullet points, max 50 words in total.

EXAMPLE (for the SoundBuds earbuds):
Your music, your way!
- 🔋 Battery life: up to 30 hours with the case
- 🎧 Noise cancelling: block out the busy world
- 📱 Bluetooth 5.2: pairs instantly with any phone

NOW WRITE IT FOR: PulseOne Mini smartwatch.
Facts to use (and nothing else): battery up to 5 days; fitness tracking of steps, heart rate, sleep and 20 workout modes; Bluetooth 5.3, works with iPhone and Android.
Bullet 1 must start with "🔋 Battery life:", bullet 2 with "💪 Fitness tracking:", bullet 3 with "📱 Bluetooth:"."""
out_5_v4 = run(prompt_5_v4, max_new_tokens=100)
evaluate_blurb(out_5_v4)

🔋 Battery life: up to 5 days on a single charge
💪 Fitness tracking: monitor steps, heart rate, sleep, and 20 workout modes
📱 Bluetooth: connect seamlessly with iPhone and Android devices

[32 words, 9s]


{'words': 29,
 '≤ 50 words': True,
 'bullet lines': 3,
 'mentions battery life': True,
 'mentions fitness tracking': True,
 'mentions Bluetooth compatibility': True,
 'friendly markers': 2,
 'PASS': True}

In [20]:
pd.DataFrame({"prompt v1 (basic)": evaluate_blurb(out_5_v1), "prompt v2 (strict)": evaluate_blurb(out_5_v2),
              "prompt v3 (strict + facts)": evaluate_blurb(out_5_v3), "prompt v4 (one-shot + facts)": evaluate_blurb(out_5_v4)})

,prompt v1 (basic),prompt v2 (strict),prompt v3 (strict + facts),prompt v4 (one-shot + facts)
words,64,32,29,29
≤ 50 words,False,True,True,True
bullet lines,0,3,0,3
mentions battery life,True,True,False,True
mentions fitness tracking,True,True,True,True
mentions Bluetooth compatibility,True,True,False,True
friendly markers,8,1,6,2
PASS,False,True,False,True


**Evaluation of the runs:**

| | v1 basic | v2 strict | v3 strict + facts | v4 one-shot + facts |
|---|---|---|---|---|
| ≤ 50 words | ❌ 64 | ✅ 32 | ✅ 29 | ✅ 29 |
| bullet format | ❌ a paragraph | ✅ 3 bullets | ❌ a single line | ✅ 3 emoji bullets |
| all 3 features | ✅ | ✅ | ❌ (no "battery", no "Bluetooth") | ✅ |
| friendly tone | ✅ very ("your personal health companion…") | ⚠️ neutral, no opening line | ✅ but over the top | ⚠️ positive ("connect seamlessly") but the opening line was skipped |
| invented facts | "12 hours" battery | "7 days", "hands-free calls" | "never need to recharge again" | none (only the given facts) |

**Lessons:**
1. A soft request ("keep it under 50 words") is treated as a suggestion. **Hard, numbered rules** ("EXACTLY 3 bullet points", "Do not exceed 50 words", one feature per bullet) fixed length and structure in v2.
2. Without the facts, the model **invents specifications** (12 hours, 7 days, hands-free calls), even when told not to. A marketing email cannot promise unverified features, so the facts must be **supplied**.
3. For a small model, **more rules can make things worse** (v3 broke the format). A **one-shot example** of the exact format, plus fixed bullet openers, was the most reliable (v4): every hard constraint was met with only true facts, although the model still skipped the friendly opening line.

The best prompt combines the hard constraints (v2), the facts (v3) and an example (v4). Always validate the output automatically (word count, features, bullets) as done above.

## 🌟 Exercise 6: Spotting and mitigating hallucinations
A short sample article stands in for the peer-reviewed paper. It deliberately does **not** contain any "50 % of species extinct by 2050" figure.

In [21]:
article_marine = """Title: Ocean warming and the redistribution of marine biodiversity (sample abstract).
Sea surface temperatures have risen by about 0.9 °C since 1900. Using 30 years of survey data for 1,200 fish and
invertebrate species, the authors show that 45% of the studied species shifted their range towards the poles, at a
median speed of 59 km per decade. Coral reef habitats declined by 14% between 2009 and 2018, mainly after
marine heatwaves. Local extinctions were recorded at the warm edge of species ranges, but the study does not
estimate global extinction rates. The authors call for marine protected areas that account for future species movements."""

vague_6 = f"Summarize this article about climate change and marine biodiversity, and say how many species will go extinct by 2050.\n\n{article_marine}"
out_vague_6 = run(vague_6, max_new_tokens=250)

The article discusses the impact of ocean warming on marine biodiversity, focusing on shifts in species' ranges due to temperature changes and declines in coral reefs caused by extreme weather events like marine heatwaves. It highlights that 45% of the surveyed species have moved poleward over the past century, with an average migration rate of 59 kilometers per decade. Additionally, it notes that local extinctions have been observed along the edges of these new ranges. However, the study does not provide estimates for global extinction rates. The researchers emphasize the need for marine protected areas that consider future species migrations to preserve biodiversity.

[103 words, 22s]


In [22]:
grounded_6 = f"""You are a careful science editor. Summarize the article below for a policy briefing.

Rules:
- Use ONLY information explicitly stated in the article. Do not use outside knowledge.
- Every number you write must appear in the article; copy it exactly.
- After each bullet, quote the supporting sentence from the article in brackets, e.g. [“...”].
- If a question cannot be answered from the article, write exactly: "Not stated in the article."

Output: 3 bullet points, then answer this question: How many marine species are projected to go extinct by 2050?

ARTICLE:
\"\"\"{article_marine}\"\"\""""
out_grounded_6 = run(grounded_6, max_new_tokens=350)

**Summary:**

The article discusses how rising sea surface temperatures due to climate change are causing marine species to shift their ranges poleward and affecting coral reefs. It mentions that 45% of the studied species have moved towards the poles over the past three decades, with a median speed of 59 kilometers per decade. Coral reef habitats have also been declining, particularly after severe marine heatwaves. However, the study does not provide estimates on global extinction rates.

**Question:** How many marine species are projected to go extinct by 2050?

**Answer:** Not stated in the article.

[94 words, 24s]


In [23]:
verify_6 = f"""Fact-check the following claim against the article.
Claim: "Over 50% of marine species are projected to go extinct by 2050."
Answer with one label - SUPPORTED, CONTRADICTED or NOT IN ARTICLE - followed by the exact sentence from the article
that justifies your label (or "none"). Only include claims explicitly stated in the article; if uncertain, say the
information is not available.

ARTICLE:
\"\"\"{article_marine}\"\"\""""
out_verify_6 = run(verify_6, max_new_tokens=120)

SUPPORTED

The article states: "Using 30 years of survey data for 1,200 fish and invertebrate species, the authors show that 45% of the studied species shifted their range towards the poles." This directly supports the claim that over 50% of marine species are projected to go extinct by 2050.

[49 words, 15s]


In [24]:
art_nums = numbers(article_marine)
for name, out in [("vague", out_vague_6), ("grounded", out_grounded_6)]:
    extra = sorted(numbers(out) - art_nums)
    print(f"{name:9s}: numbers not in the article: {extra} | says 'not stated': {'not stated' in out.lower()}")

vague    : numbers not in the article: [] | says 'not stated': False
grounded : numbers not in the article: [2050.0] | says 'not stated': True


### 1–2. Prompts
- **Precise anti-hallucination prompt** (`grounded_6`): *"Use ONLY information explicitly stated in the article. Do not use outside knowledge. Every number must appear in the article… If a question cannot be answered from the article, write exactly: 'Not stated in the article.'"*
- **Verification-language prompt** (`verify_6`): asks the model to label the claim SUPPORTED / CONTRADICTED / NOT IN ARTICLE with the exact justifying sentence, and adds *"Only include claims explicitly stated in the article; if uncertain, say the information is not available."*

**What happened:**
- With the grounded prompt, the model answered the trap question correctly ("**Not stated in the article**"), kept the right figures (45 %, 59 km per decade), and introduced no number absent from the article. 2050 appears only because it repeats the question. It did, however, ignore the request to quote supporting sentences.
- The **vague** prompt avoided the 50 % claim but made a subtler mistake: "over the **past century**" instead of 30 years of data. This kind of distortion is easy to miss.
- The **verification prompt failed**: the 1.5 B model labelled the false claim **SUPPORTED** and cited an unrelated sentence about range shifts. Prompting alone does not make a small model a reliable fact-checker.

### 3. Mitigation strategies used
1. **Grounding and scope restriction:** the source text is given inside delimiters, with "use ONLY the article", "no outside knowledge" and "every number must appear in the text".
2. **An explicit abstention path:** a fixed answer ("Not stated in the article") when the information is missing, so the model is not pushed to invent an answer.
3. *(bonus)* **Verification and citation requirements:** quote the supporting sentence, and run a separate claim-checking pass. Combined with an automatic check that every number in the output appears in the source (the `numbers()` check above), and ideally a stronger model or an NLI fact-checker, since the small model's self-verification failed.

### 4. Domains where hallucinations are especially risky
- **Healthcare / medicine:** an invented dosage, drug interaction or study result can directly harm patients. Clinicians and patients may trust a fluent, confident answer, and errors propagate into treatment decisions.
- **Legal:** invented case law or statutes (as in real cases where lawyers were sanctioned for citing fake ChatGPT precedents) can lose a case, mislead clients and break professional duties. Legal outcomes depend on exact wording and real precedent.
- *(also)* **Finance and journalism / science communication:** fabricated figures move investment decisions or spread misinformation, as with the "50 % of species extinct by 2050" claim, which would misinform policy makers if published.